# Choosing the number of factors — the formal tests

Three tests from `ICtest` for the number of independent components in the
significant coefficient matrix, which is what `03_FactorizeBetaMatrix` then
decomposes.

:::{warning}
`ICtest` is loaded here and nowhere else. Once attached it registers an `Ops`
method that collides with ggplot2 4.x's S7 objects and breaks every `+` chain
for the rest of the session — and detaching it does not undo that. The
explained-variance sweep, which is drawn with ggplot, is therefore a separate
notebook, `02`.
:::

## Setup

In [ ]:
library(ICtest)

source("../paths.R")   # SCREEN_DATA, REFERENCE, TEXTFILES; see data/README.md

selCoefs <- read.csv(referencePath("SignificantCoefMatrix.csv"), row.names = 1)
selCoefs <- t(selCoefs)
colnames(selCoefs) <- sapply(colnames(selCoefs), function(x) strsplit(x, "_")[[1]][2])
dim(selCoefs)

## Asymptotic test

FOBI with the asymptotic null distribution, up to 20 components.

In [ ]:
FOBIasymp(selCoefs,k = 20, type = "S1", model = "ICA")

## Bootstrap test

The same, with 200 bootstrap replicates.

In [ ]:
FOBIboot(X=selCoefs, k=20, n.boot = 200, s.boot = "B1")

## The ladle estimator

Combines the eigenvalues with the bootstrap variability of the
eigenvectors; the minimum of the ladle is the estimate.

In [ ]:
test <- FOBIladle(selCoefs)
test
summary(test)


In [ ]:
# ladleplot() is base graphics and fails on the IRkernel device with "figure
# margins too large" whatever repr.plot.* is set to, so it goes to a PDF.
# plot(test) is a different, larger multi-panel figure that will not fit on
# any device tried here, so only the ladle plot is drawn.
pdf("figures/LadlePlot.pdf", width = 9, height = 7)
ladleplot(test)
dev.off()

cat("ladle estimate of the number of components:", test$k, "\n")
